In [1]:
import argparse
import os
import sys
import time
import torch
import mlflow
import pandas as pd

from datasets import Dataset #load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    AutoProcessor
)
from functools import partial

In [2]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT") #modify
USER = os.getenv("SLURM_JOB_USER") #modify

In [3]:
input_model = "google/medgemma-1.5-4b-it"
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
model_output_name = f"{input_model}_finetuned"
json_file = f"/scratch/{SLURM_JOB_ACCOUNT}/data/structured_notes_gpt_oss_120b_srun.json"
batch_size = 2
cache_dir = f"/scratch/{SLURM_JOB_ACCOUNT}/hf-cache/hub"
max_tokens = 2048
num_workers = int(os.getenv("SLURM_CPUS_PER_TASK"))

output_model_dir = os.path.join(output_path, model_output_name)
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

In [4]:
def preprocess(examples, tokenizer, max_tokens=2048):
    """Convert input_data/output pairs into tokenized chat format."""
    input_ids_list = []
    labels_list = []
    attention_mask_list = []

    for input_data, output in zip(examples["conversation"], examples["structured_note"]):

        # Build chat messages
        messages = [
            {
                "role": "system",
                "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
            {"role": "user", "content": input_data},
            {"role": "assistant", "content": output},
        ]

        # Apply chat template — tokenize full conversation
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        # Also build prompt-only part to know where assistant response starts
        prompt_messages = messages[:-1]
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # Tokenize full conversation
        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
        )

        # Tokenize prompt only to get its length
        prompt_tokenized = tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        prompt_len = len(prompt_tokenized["input_ids"])

        # Mask prompt tokens in labels — only compute loss on assistant response
        labels = [-100] * prompt_len + input_ids[prompt_len:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }

In [5]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [6]:
tokenizer = AutoTokenizer.from_pretrained(input_model, use_fast=True, cache_dir=cache_dir)
processor = AutoProcessor.from_pretrained(input_model, cache_dir=cache_dir)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    input_model,
    torch_dtype=torch.bfloat16,
    device_map=device,
    cache_dir=cache_dir
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
peft_config = LoraConfig(
    lora_alpha=8,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [9]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 38,497,792 || all params: 4,338,577,264 || trainable%: 0.8873


In [10]:
## DATA
df = pd.read_json(json_file)[:2000]
dataset = Dataset.from_pandas(df, preserve_index=False)
split = dataset.train_test_split(test_size=0.05, seed=42)

raw_train = split["train"]
raw_val = split["test"]

print(f"  Train size: {len(raw_train)}")
print(f"  Val size:   {len(raw_val)}")

  Train size: 1900
  Val size:   100


In [11]:
preprocess_fn = partial(preprocess, tokenizer=tokenizer, max_tokens=max_tokens)

In [12]:
tokenized_train = raw_train.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_train.column_names,
    num_proc=num_workers,
)

Map (num_proc=7):   0%|          | 0/1900 [00:00<?, ? examples/s]

In [13]:
tokenized_val = raw_val.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_val.column_names,
    num_proc=num_workers,
)

Map (num_proc=7):   0%|          | 0/100 [00:00<?, ? examples/s]

In [14]:
training_args = TrainingArguments(
    disable_tqdm=False,
    output_dir=output_model_dir,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True,
    load_best_model_at_end=True,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size*4,
    dataloader_num_workers=num_workers,
    ddp_find_unused_parameters=True,
    dataloader_pin_memory=True,
    save_safetensors=True,
    metric_for_best_model="eval_loss",
    eval_strategy="steps",
    eval_steps=300,
    num_train_epochs=1,
    report_to=["mlflow"],
    logging_steps=50,
    logging_strategy="steps",
    run_name=f"{model_output_name}_{os.environ.get('SLURM_JOB_ID', 'local')}",
)

In [15]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/tmp/ipykernel_68830/1846427416.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/opt/venv/lib/python3.12/site-packages/deepspeed/accelerator/cuda_accelerator.py:36: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/opt/venv/lib/python3.12/site-packages/apex/transformer/functional/fused_rope.py:54: UserWarning: Using the native apex kernel for RoPE.
  warnings.warn("Using the native apex kernel for RoPE.", UserWarning)


In [17]:
start_train = time.time()

trainer.train()

stop_train = time.time()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.
/opt/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Step,Training Loss,Validation Loss
300,1.298400,1.260723
600,1.203300,1.215090
900,1.213700,1.201459


In [18]:
elapsed = stop_train - start_train
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"Training took: {hours}h {minutes}m {seconds}s")

Training took: 0h 19m 50s


In [19]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained(
    merged_output_dir,
    safe_serialization=False
)

tokenizer.save_pretrained(merged_output_dir)
processor.save_pretrained(merged_output_dir)

['/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged/processor_config.json']

# Testing the finetuned model

In [4]:
prompt = """Doctor: Hey, how are you doing today?

Patient: Hello doctor. I am feeling pain on the bottom right in my belly.

Doctor: How long has the pain been there?

Patient: It started yesterday evening and got worse during the night.

Doctor: Can you describe the pain? Is it sharp, dull, cramping, or something else?

Patient: It started as a dull ache, but now it feels sharp when I move or walk.

Doctor: On a scale from 1 to 10, how strong is the pain?

Patient: Around 7 out of 10.

Doctor: Have you noticed any nausea, vomiting, fever, or changes in appetite?

Patient: Yes, I feel nauseous and I did not want breakfast this morning. I also think I have a slight fever.

Doctor: Have you had diarrhea or constipation?

Patient: No diarrhea, but I have not gone to the bathroom since yesterday.

Doctor: Does anything make the pain better or worse?

Patient: Moving makes it worse. Lying still helps a little.

Doctor: Have you experienced this kind of pain before?

Patient: No, never this bad.

Doctor: Do you have any medical conditions or take any medications regularly?

Patient: No major medical conditions. I only take allergy medicine sometimes.

Doctor: Thank you. I would like to examine your abdomen now, especially the lower right side.

Patient: Okay.

Doctor: When I press here, does it hurt?

Patient: Yes, especially when you let go.

Doctor: I understand. Based on your symptoms and the examination, this could be appendicitis. I recommend blood tests and an abdominal scan as soon as possible.

Patient: Is it serious?

Doctor: It can become serious if untreated, but we caught it early. We will arrange further testing immediately.

Patient: Thank you, doctor.

Doctor: You're welcome. We will take good care of you."""

In [12]:
def build_messages(example: str) -> list[dict]:
    return [
        {"role": "system",
        "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
        {"role": "user",   "content": example},
    ]

In [6]:
from vllm import LLM, SamplingParams

In [7]:
SAMPLING = dict(temperature=0.0, max_tokens=1000)

In [8]:
llm = LLM(model=merged_output_dir, tensor_parallel_size=1, dtype="bfloat16")

INFO 05-29 09:12:57 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged'}
INFO 05-29 09:12:57 [model.py:555] Resolved architecture: Gemma3ForConditionalGeneration
INFO 05-29 09:12:57 [model.py:1680] Using max model len 131072
INFO 05-29 09:12:57 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-29 09:12:57 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-29 09:12:57 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


The tokenizer you are loading from '/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


INFO 05-29 09:12:59 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-29 09:12:59 [nixl_utils.py:34] NIXL is not available
WARNING 05-29 09:12:59 [nixl_utils.py:44] NIXL agent config is not available


/opt/venv/lib/python3.12/site-packages/tensorizer/utils.py:23: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The tokenizer you are loading from '/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading thi

WARNING 05-29 09:13:08 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=107397) INFO 05-29 09:13:18 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged', speculative_config=None, tokenizer='/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=True, quan

...

(EngineCore pid=107397) INFO 05-29 09:13:32 [default_loader.py:384] Loading weights took 4.37 seconds
(EngineCore pid=107397) INFO 05-29 09:13:33 [gpu_model_runner.py:4879] Model loading took 9.91 GiB memory and 4.925218 seconds
(EngineCore pid=107397) INFO 05-29 09:13:33 [gpu_model_runner.py:5820] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 32 image items of the maximum feature size.
(EngineCore pid=107397) WARNING 05-29 09:13:34 [op.py:241] Priority not set for op rms_norm, using native implementation.
(EngineCore pid=107397) INFO 05-29 09:13:41 [backends.py:1069] Using cache directory: /users/hmerilai/.cache/vllm/torch_compile_cache/6bcbde8880/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=107397) INFO 05-29 09:13:41 [backends.py:1128] Dynamo bytecode transform time: 3.89 s
(EngineCore pid=107397) INFO 05-29 09:13:44 [backends.py:290] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 3.041 s
(Engine

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 14.72it/s]
Capturing CUDA graphs (decode, FULL):   0%|          | 0/35 [00:00<?, ?it/s]

(EngineCore pid=107397) WARNING 05-29 09:13:50 [chunked_prefill_paged_decode.py:400] Cannot use ROCm custom paged attention kernel, falling back to Triton implementation.


Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 14.27it/s]


(EngineCore pid=107397) INFO 05-29 09:13:52 [gpu_model_runner.py:6133] Graph capturing finished in 7 secs, took 0.41 GiB
(EngineCore pid=107397) INFO 05-29 09:13:52 [core.py:299] init engine (profile, create kv cache, warmup model) took 19.60 s (compilation: 7.53 s)
(EngineCore pid=107397) INFO 05-29 09:13:53 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


In [9]:
sampling_params = SamplingParams(**SAMPLING)

In [13]:
outputs = llm.chat(build_messages(prompt), sampling_params, use_tqdm=True)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [14]:
print(outputs[0].outputs[0].text)

**REASON FOR VISIT:**  
The patient presented with acute right lower quadrant abdominal pain that began the previous evening and progressively worsened overnight.

**PATIENT DETAILS AND HISTORY:**  
The patient is a 22‑year‑old female. She reports no chronic medical conditions, regular medications, or prior surgeries. She occasionally uses over‑the‑counter allergy medication. No lifestyle factors were specified.

**CURRENT STATUS:**  
On examination, the right lower quadrant was tender to palpation, with rebound tenderness present. Vital signs were stable, and laboratory studies showed a white blood cell count of 12.5 ×�0.10 ×10⁹/L with a left shift. Imaging revealed a 1.5 cm fluid‑filled appendix with a small amount of free fluid in the right paracolic gutter.

**TREATMENTS/ACTIONS:**  
The patient was admitted to the hospital and received intravenous fluids. She underwent an appendectomy via a midline incision. Post‑operatively she was monitored in the recovery room, discharged home 